In [449]:
import pandas as pd
import numpy as np
import difflib

# require pip installs
from thefuzz import process 
# from faker import Faker
# fake = Faker()

## fuzzy matching

* this is helpful if you're trying to merge data but the values in the match column don't exactly match
* in this example, our main goal is to add one or more columns from "country_codes" to "essential_indicators"

In [450]:
df1 = pd.read_csv('data/essential_indicators_messy.csv', index_col=0)

df1.columns
list1 = df1['Country']
print(list1.shape)
list1.head()

(217,)


0       Afghanistan
1           Albania
2           Algeria
3    American Samoa
4           Andorra
Name: Country, dtype: object

In [451]:
df2 = pd.read_csv('data/country_codes.csv')#, index_col=0)
df2.head()

df2.columns
list2 = df2['name']
print(list2.shape)
list2.head()

(249,)


0       Afghanistan
1     Åland Islands
2           Albania
3           Algeria
4    American Samoa
Name: name, dtype: object

In [452]:
# count the number of exact matches
matches = list1.isin(list2).sum()
matches

np.int64(181)

In [453]:
# Find names in list1 that aren't in list2
f = ~list1.isin(list2)
missing = list1[f]
print(len(missing))
missing.head()


36


13        Bahamas, The
23             Bolivia
38     Channel Islands
43    Congo, Dem, Rep,
44         Congo, Rep,
Name: Country, dtype: object

## using the Fuzz

In [454]:
# match missing names to cc names using thefuzz
def find_closest_match_fuzz(name, choices):
    match, score = process.extractOne(name, choices)
    return match, score# if score > 80 else None

In [455]:
find_closest_match_fuzz("Bolivia", list2.to_list())

('Bolivia, Plurinational State of', 90)

In [456]:
# remove "Republic" from missing - that seems to be messing things up
missing_cleaned = missing.str.replace("Republic", "", regex=False)

In [457]:
matches, scores = zip(*[find_closest_match_fuzz(n,list2.to_list()) for n in missing_cleaned])

## inspect the matches

In [458]:

match_table = pd.DataFrame({
    'Country':missing,
    'name':matches,
    'Score':scores
})

print(len(match_table))

# see the worst matches
# match_table.sort_values('Score')
match_table.head(10)

36


,Country,name,Score
13,"Bahamas, The",Bahamas,90
23,Bolivia,"Bolivia, Plurinational State of",90
38,Channel Islands,Cocos (Keeling) Islands,86
43,"Congo, Dem, Rep,",Congo,90
44,"Congo, Rep,",Congo,90
46,Cote d'Ivoire,Côte d'Ivoire,96
49,Curacao,Curaçao,92
57,"Egypt, Arab Rep,",Egypt,90
70,"Gambia, The",Gambia,90
84,"Hong Kong SAR, China",China,90


## applying the corrections

In [459]:
# match_dict = {}
# for i in range(len(missing)):
#     k = missing.iloc[i]
#     v = matches[i]
#     match_dict[k]=v

name_corrections = {missing.iloc[i]:matches[i] for i in range(len(missing))}
# match_dict



In [460]:
# now we can apply these corrections to df and merge again
df1['Country'] = df1['Country'].replace(name_corrections)
df1['Country']

0                                 Afghanistan
1                                     Albania
2                                     Algeria
3                              American Samoa
4                                     Andorra
                        ...                  
212                                    Zambia
213                                  Zimbabwe
214                  Virgin Islands (British)
215                                 Gibraltar
216    Korea, Democratic People's Republic of
Name: Country, Length: 217, dtype: object

## finishing touches

In [461]:
sus = [38,43,44, 84]

matches = {}

for i in sus:
    n = match_table.loc[i,'Country']
    bests = process.extractBests(n,list2.to_list())
    matches[i] = [n] + [b[0] for b in bests]

fix_it_table = pd.DataFrame(matches).T
fix_it_table

,0,1,2,3,4,5
38,Channel Islands,Cocos (Keeling) Islands,Falkland Islands (Malvinas),Heard Island and McDonald Islands,Northern Mariana Islands,South Georgia and the South Sandwich Islands
43,"Congo, Dem, Rep,",Congo,"Congo, Democratic Republic of the",Togo,Tonga,Mongolia
44,"Congo, Rep,",Congo,"Congo, Democratic Republic of the",Togo,Tonga,Dominican Republic
84,"Hong Kong SAR, China",China,Hong Kong,India,Chile,Congo


In [462]:
# make a choice

map_it = {
    38: None,
    43: 2,
    44: 1,
    84: 2
}

for k,v in map_it.items():
    if v:
        match_table.loc[k,'name'] = fix_it_table.loc[k][v]
    else:
        match_table.loc[k,'name'] = None

match_table


,Country,name,Score
13,"Bahamas, The",Bahamas,90
23,Bolivia,"Bolivia, Plurinational State of",90
38,Channel Islands,None,86
43,"Congo, Dem, Rep,","Congo, Democratic Republic of the",90
44,"Congo, Rep,",Congo,90
46,Cote d'Ivoire,Côte d'Ivoire,96
49,Curacao,Curaçao,92
57,"Egypt, Arab Rep,",Egypt,90
70,"Gambia, The",Gambia,90
84,"Hong Kong SAR, China",Hong Kong,90
